# Backpropagation and automatic differentiation

Training needs the gradient of the loss with respect to every weight. Backpropagation is the chain rule applied layer by layer, from the output back to the input. Automatic differentiation is the general machine that does it for any computation. Section 01, `04_autodiff.ipynb`, built a small autodiff engine. Here we do backprop by hand on a tiny network, check it against Keras and torch, then build a Keras functional-API model that no `Sequential` stack can express.

Order: intuition, maths, cost function, from-scratch version, library version, tuning, what breaks.

In [ ]:
from common.dl import set_seed, setup_keras

setup_keras("torch")
import keras
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from common.datasets import load_california_housing

SEED = 42
set_seed(SEED)
keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=4, suppress=True)

## 1. Intuition

The forward pass turns inputs into a loss. Each step on the way is a simple function with a simple derivative. The loss changes with a weight only through the chain of steps between them, so the gradient is a product of the local derivatives along that chain.

The trick is order. Work from the loss backwards and reuse each result. One backward pass costs about as much as one forward pass, whether the network has ten weights or ten million. Trying each weight in turn would cost one forward pass per weight.

## 2. Maths and cost function

Take one hidden layer, a tanh activation and a squared-error loss:

$$\mathbf h=\tanh(\mathbf z_1),\; \mathbf z_1=XW_1+\mathbf b_1,\qquad \hat y=\mathbf hW_2+b_2,\qquad J=\frac1n\sum(\hat y-y)^2$$

Backwards, one line per step:

$$\delta_2=\frac{2}{n}(\hat y-y),\quad \frac{\partial J}{\partial W_2}=\mathbf h^\top\delta_2,\quad \frac{\partial J}{\partial b_2}=\textstyle\sum\delta_2$$

$$\delta_1=(\delta_2W_2^\top)\odot(1-\mathbf h^2),\quad \frac{\partial J}{\partial W_1}=X^\top\delta_1,\quad \frac{\partial J}{\partial \mathbf b_1}=\textstyle\sum\delta_1$$

The factor $1-\mathbf h^2$ is the tanh derivative. It appears at every layer, and it is why deep stacks of saturating units starve the early layers of gradient (notebook 05).

## 3. From scratch

A network with 3 inputs, 4 tanh hidden units and 1 output, on 6 rows. The backward pass is five lines.

In [ ]:
X = rng.normal(size=(6, 3))
y = rng.normal(size=(6, 1))
W1, b1 = rng.normal(size=(3, 4)) * 0.5, rng.normal(size=4) * 0.1
W2, b2 = rng.normal(size=(4, 1)) * 0.5, rng.normal(size=1) * 0.1


def loss_and_grads(X, y, W1, b1, W2, b2, tanh_derivative=True):
    h = np.tanh(X @ W1 + b1)
    y_hat = h @ W2 + b2
    loss = np.mean((y_hat - y) ** 2)
    d2 = 2 * (y_hat - y) / len(X)
    d1 = (d2 @ W2.T) * ((1 - h**2) if tanh_derivative else 1.0)
    return loss, [X.T @ d1, d1.sum(axis=0), h.T @ d2, d2.sum(axis=0)]


loss, grads = loss_and_grads(X, y, W1, b1, W2, b2)
print("loss:", round(loss, 4))
print("dJ/dW2 by hand:", grads[2].ravel())

Three independent checks. First, a numerical gradient (nudge each weight by a tiny amount, as in section 01). Second, torch autograd in float64. Third, Keras: a model with the same weights, with the gradient read off its variables after a torch backward pass.

In [ ]:
def numerical_grad(param_index: int, eps: float = 1e-6) -> np.ndarray:
    params = [W1.copy(), b1.copy(), W2.copy(), b2.copy()]
    g = np.zeros_like(params[param_index])
    for idx in np.ndindex(g.shape):
        up, down = [p.copy() for p in params], [p.copy() for p in params]
        up[param_index][idx] += eps
        down[param_index][idx] -= eps
        g[idx] = (loss_and_grads(X, y, *up)[0] - loss_and_grads(X, y, *down)[0]) / (2 * eps)
    return g


# torch autograd, float64
tp = [torch.tensor(a, dtype=torch.float64, requires_grad=True) for a in (W1, b1, W2, b2)]
h_t = torch.tanh(torch.tensor(X) @ tp[0] + tp[1])
((h_t @ tp[2] + tp[3] - torch.tensor(y)) ** 2).mean().backward()

# Keras model with the same weights
set_seed(SEED)
net = keras.Sequential(
    [keras.Input((3,)), keras.layers.Dense(4, activation="tanh"), keras.layers.Dense(1)]
)
net.layers[0].set_weights([W1, b1])
net.layers[1].set_weights([W2, b2])
device = next(net.parameters()).device
out = net(torch.tensor(X, dtype=torch.float32, device=device))
(((out - torch.tensor(y, dtype=torch.float32, device=device)) ** 2).mean()).backward()
keras_grads = [v.value.grad.cpu().numpy() for v in net.trainable_weights]

for i, name in enumerate(["W1", "b1", "W2", "b2"]):
    hand = grads[i]
    print(
        f"{name}: vs numerical {np.abs(hand - numerical_grad(i)).max():.1e}, "
        f"vs torch {np.abs(hand - tp[i].grad.numpy()).max():.1e}, "
        f"vs keras {np.abs(hand - keras_grads[i]).max():.1e}"
    )

All three agree. The numerical check is good to about 1e-10, limited by the nudge size. Torch in float64 matches to 1e-17, which is exact for practical purposes. Keras works in float32, so it differs by about 1e-8, the precision of the numbers themselves. One trap: read Keras gradients through `trainable_weights`. The order of `parameters()` differs, and matching by position gave errors near 0.1 on the first attempt.

## 4. Library version: the functional API

`Sequential` is a straight line of layers. The functional API treats layers as functions on tensors, so you can branch, merge and take several inputs. Wide-and-deep is the standard example. A deep path learns complex patterns, and a short wide path lets the model use simple features directly. We feed the wide path 5 features and the deep path 6, with some overlap. Gradients for all branches come from the same backward pass.

In [ ]:
Xh, yh = load_california_housing(6000, seed=SEED)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(Xh.to_numpy(), yh.to_numpy(), random_state=SEED)
scaler = StandardScaler().fit(Xh_tr)
Xs_tr, Xs_te = scaler.transform(Xh_tr), scaler.transform(Xh_te)
WIDE, DEEP = slice(0, 5), slice(2, 8)


def wide_and_deep(seed: int = SEED) -> keras.Model:
    set_seed(seed)
    wide_in, deep_in = keras.Input((5,), name="wide"), keras.Input((6,), name="deep")
    x = keras.layers.Dense(32, activation="relu")(deep_in)
    x = keras.layers.Dense(32, activation="relu")(x)
    merged = keras.layers.Concatenate()([wide_in, x])
    model = keras.Model([wide_in, deep_in], keras.layers.Dense(1)(merged))
    model.compile(optimizer="adam", loss="mse")
    return model


def plain_deep(seed: int = SEED) -> keras.Model:
    set_seed(seed)
    model = keras.Sequential(
        [
            keras.Input((8,)),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(32, activation="relu"),
            keras.layers.Dense(1),
        ]
    )
    model.compile(optimizer="adam", loss="mse")
    return model


model = wide_and_deep()
print("parameters:", model.count_params())
model.fit([Xs_tr[:, WIDE], Xs_tr[:, DEEP]], yh_tr, epochs=40, batch_size=512, verbose=0)
print("test MSE:", round(model.evaluate([Xs_te[:, WIDE], Xs_te[:, DEEP]], yh_te, verbose=0), 3))

## 5. Tuning

Is wide-and-deep better than the plain stack? One run cannot say, because the random start changes the answer. Repeat with three seeds.

In [ ]:
scores = {"plain": [], "wide+deep": []}
for seed in (0, 1, 2):
    p = plain_deep(seed)
    p.fit(Xs_tr, yh_tr, epochs=40, batch_size=512, verbose=0)
    scores["plain"].append(p.evaluate(Xs_te, yh_te, verbose=0))
    w = wide_and_deep(seed)
    w.fit([Xs_tr[:, WIDE], Xs_tr[:, DEEP]], yh_tr, epochs=40, batch_size=512, verbose=0)
    scores["wide+deep"].append(w.evaluate([Xs_te[:, WIDE], Xs_te[:, DEEP]], yh_te, verbose=0))
for name, s in scores.items():
    print(f"{name:>10}: mean {np.mean(s):.3f}, spread {np.ptp(s):.3f}, runs {np.round(s, 3)}")

The plain stack is ahead, 0.41 against 0.46, but the gap (0.05) is about the size of the spread across seeds (0.04 to 0.06), so treat it as a tie that leans plain. The wide path adds little on this data, and the deep path sees fewer features than the plain model does. Wide-and-deep earns its place when the simple features carry signal that the deep path would wash out, or when inputs differ in kind, such as text plus numbers.

## 6. What breaks

Backprop code fails silently. A forgotten derivative still trains, just badly. Drop the tanh factor and compare against the numerical gradient.

In [ ]:
_, wrong = loss_and_grads(X, y, W1, b1, W2, b2, tanh_derivative=False)
for i, name in ((0, "W1"), (2, "W2")):
    num = numerical_grad(i)
    rel = np.linalg.norm(wrong[i] - num) / np.linalg.norm(num)
    print(f"{name} relative error without tanh derivative: {rel:.2e}")

The output layer gradient `W2` is still exact, because the bug sits upstream of it. `W1` is wrong by a large fraction of its size. A relative error near 1e-7 or below passes a gradient check. Anything above 1e-3 is a bug. Run this check once on every hand-written layer.